# 13_02_ReAct_ループとツール呼び出しの最小構成

コード：`ch13/blocks/13_02_ReAct_ループとツール呼び出しの最小構成.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


## 本文の掲載コード（参照）


```python
import json, time

def run_agent(user_msg, *, max_steps=12, budget_usd=0.50, deadline_s=60):
    messages = [{"role": "user", "content": user_msg}]
    seen, spent, t0 = set(), 0.0, time.time()

    for step in range(max_steps):                       # ★ ①ステップ上限
        if spent > budget_usd:        return stop("コスト上限", messages)
        if time.time() - t0 > deadline_s: return stop("時間切れ", messages)

        resp   = llm(messages, tools=TOOLS)
        spent += resp.cost
        messages.append({"role": "assistant", "content": resp.content})

        calls = [c for c in resp.content if c["type"] == "tool_use"]
        if not calls:
            return {"answer": resp.text, "steps": step, "cost": spent}

        results = []
        for c in calls:
            key = (c["name"], json.dumps(c["input"], sort_keys=True))
            if key in seen:                             # ★ ②同一呼び出しの検出
                results.append(err(c, "同じ呼び出しを繰り返しています。"
                                      "別の方法を試すか、分からないと答えてください。"))
                continue
            seen.add(key)
            results.append(execute(c))

        messages.append({"role": "user", "content": results})

    return stop("ステップ上限", messages)                 # ★ ③打ち切りも「結果」

def execute(call):
    name, args = call["name"], call["input"]
    try:
        if name in NEEDS_APPROVAL and not approved(call):
            return err(call, "承認待ちです。ユーザーに確認してください。")
        out = TOOL_IMPL[name](**args)
        text = json.dumps(out, ensure_ascii=False)[:4000]   # ★ 出力を必ず切り詰める
    except Exception as e:
        text = f"ERROR: {type(e).__name__}: {e}"            # ★ 例外も文字列で返す
    return {"type": "tool_result", "tool_use_id": call["id"], "content": text}

def err(call, msg):                 # 失敗も tool_result としてモデルに返す
    return {"type": "tool_result", "tool_use_id": call["id"], "content": msg, "is_error": True}

def stop(reason, messages):         # 打ち切りの理由を、そのまま結果にする
    return {"answer": None, "stopped": reason, "steps": len(messages)}
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "13", "--name=agent", "--skip-docker"], cwd=CODE, check=True)
